# Free cash flow screens from XBRL financials

Screen companies by revenue, free cash flow and margins with one query, then rank them in pandas.

Read it with charts at https://www.quantisnow.com/sec-api/guides/free-cash-flow-screen. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

The financials endpoint has `cashFlow.freeCashFlow` and `incomeStatement.revenue` on every period with the same names for every company, so a screen is a range query plus a sort.

## Large companies with negative free cash flow

### Which non-financial companies with over $10 billion in revenue had negative free cash flow last fiscal year?

A screen across every filer in one call, which needs the same field names for every company to work.

```shell
curl -X POST https://www.quantisnow.com/api/v1/financials \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "fiscalPeriod:FY AND fiscalYear:2025 AND incomeStatement.revenue:>10000000000 AND cashFlow.freeCashFlow:<0 AND currency:USD AND NOT sector:Finance",
  "from": 0,
  "size": 5,
  "sort": [{"incomeStatement.revenue": {"order": "desc"}}]
}'
```

Response (real, first 2 of 40):

```json
{
  "total": {
    "value": 40,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0000012927:2025:FY",
      "cik": "0000012927",
      "ticker": "BA",
      "companyName": "Boeing Company",
      "fiscalYear": 2025,
      "fiscalPeriod": "FY",
      "periodStart": "2025-01-01",
      "periodEnd": "2025-12-31",
      "filedOn": "2026-01-30",
      "form": "10-K",
      "accessionNo": "0001628280-26-004357",
      "sector": "Industrials",
      "currency": "USD",
      "derived": false,
      "incomeStatement": {
        "revenue": 89463000000,
        "costOfRevenue": 85174000000,
        "grossProfit": 4289000000,
        "researchDevelopment": 3615000000,
        "sellingGeneralAdmin": 6090000000,
        "operatingIncome": 4281000000,
        "interestExpense": null,
        "pretaxIncome": 2635000000,
        "incomeTax": 397000000,
        "netIncome": 2235000000,
        "epsBasic": 2.49,
        "epsDiluted": 2.48,
        "sharesDiluted": 762300000
      },
      "balanceSheet": {
        "cash": 10921000000,
        "shortTermInvestments": 18479000000,
        "receivables": 2921000000,
        "inventory": null,
        "currentAssets": 128459000000,
        "goodwill": 17275000000,
        "totalAssets": 168235000000,
        "currentLiabilities": 108115000000,
        "shortTermDebt": 8461000000,
        "longTermDebt": 45637000000,
        "totalDebt": 54098000000,
        "totalLiabilities": 162778000000,
        "equity": 5454000000
      },
      "cashFlow": {
        "operatingCashFlow": 1065000000,
        "investingCashFlow": 499000000,
        "financingCashFlow": -3763000000,
        "capex": 2942000000,
        "freeCashFlow": -1877000000,
        "dividendsPaid": 331000000,
        "shareRepurchases": null,
        "stockCompensation": 426000000,
        "depreciation": 1953000000
      }
    },
    {
      "id": "0001001085:2025:FY",
      "cik": "0001001085",
      "ticker": "BN",
      "companyName": "Brookfield Corporation",
      "fiscalYear": 2025,
      "fiscalPeriod": "FY",
      "periodStart": "2025-01-01",
      "periodEnd": "2025-12-31",
      "filedOn": "2026-03-18",
      "form": "40-F",
      "accessionNo": "0001001085-26-000006",
      "sector": "Real Estate",
      "currency": "USD",
      "derived": false,
      "incomeStatement": {
        "revenue": 75100000000,
        "costOfRevenue": 28351000000,
        "grossProfit": 46749000000,
        "researchDevelopment": null,
        "sellingGeneralAdmin": 5131000000,
        "operatingIncome": null,
        "interestExpense": null,
        "pretaxIncome": null,
        "incomeTax": 1135000000,
        "netIncome": 1307000000,
        "epsBasic": 0.51,
        "epsDiluted": 0.49,
        "sharesDiluted": 2367800000
      },
      "balanceSheet": {
        "cash": 16242000000,
        "shortTermInvestments": null,
        "receivables": 22772000000,
        "inventory": 5923000000,
        "currentAssets": null,
        "goodwill": 43355000000,
        "totalAssets": 518971000000,
        "currentLiabilities": null,
        "shortTermDebt": null,
        "longTermDebt": 13800000000,
        "totalDebt": 13800000000,
        "totalLiabilities": 352777000000,
        "equity": 166194000000
      },
      "cashFlow": {
        "operatingCashFlow": 10958000000,
        "investingCashFlow": -31777000000,
        "financingCashFlow": 21687000000,
        "capex": 14752000000,
        "freeCashFlow": -3794000000,
        "dividendsPaid": null,
        "shareRepurchases": null,
        "stockCompensation": null,
        "depreciation": 10379000000
      }
    }
  ]
}
```

## Rank by FCF margin in Python

In [ ]:
import os, requests, pandas as pd

rows, start = [], 0
while True:
    page = requests.post("https://www.quantisnow.com/api/v1/financials",
        headers={"Authorization": f"Bearer {os.environ['QUANTISNOW_API_KEY']}"},
        json={"query": "fiscalPeriod:FY AND fiscalYear:2025 AND incomeStatement.revenue:>1000000000 AND currency:USD",
              "size": 50, "from": start}).json()
    rows += page["data"]
    start += 50
    if start >= min(page["total"]["value"], 1000):
        break

df = pd.json_normalize(rows)
df["fcf_margin"] = df["cashFlow.freeCashFlow"] / df["incomeStatement.revenue"]
print(df.sort_values("fcf_margin", ascending=False)[["ticker", "companyName", "fcf_margin"]].head(25))

## From the command line

```shell
npx quantisnow query financials 'fiscalPeriod:FY AND fiscalYear:2025 AND cashFlow.freeCashFlow:<0 AND incomeStatement.revenue:>10000000000' -n 50
```

Related: [XBRL financial data](https://www.quantisnow.com/sec-api/guides/xbrl-financial-data-api), [Share count over time](https://www.quantisnow.com/sec-api/guides/share-dilution-over-time).